In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df_missing = pd.read_csv('data/processed/phishing_missing.csv')

X_missing = df_missing.drop('Result', axis=1)
y = df_missing['Result']

print(f"Liczba braków w X_missing przed imputacją:\n{X_missing.isnull().sum().head(5)}")

Liczba braków w X_missing przed imputacją:
having_IP_Address           1105
URL_Length                  1105
Shortining_Service          1105
having_At_Symbol            1105
double_slash_redirecting    1105
dtype: int64


In [4]:
from sklearn.impute import SimpleImputer, KNNImputer


# Sposób 1: Most Frequent
imputer_mode = SimpleImputer(strategy='most_frequent')
X_imputed_mode = pd.DataFrame(imputer_mode.fit_transform(X_missing), columns=X_missing.columns)

# Sposób 2: Constant
imputer_constant = SimpleImputer(strategy='constant', fill_value=0)
X_imputed_const = pd.DataFrame(imputer_constant.fit_transform(X_missing), columns=X_missing.columns)

# Sposób 3: KNN
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed_knn_raw = imputer_knn.fit_transform(X_missing)
X_imputed_knn = pd.DataFrame(np.round(X_imputed_knn_raw), columns=X_missing.columns)

df_mode = pd.concat([X_imputed_mode, y], axis=1)
df_const = pd.concat([X_imputed_const, y], axis=1)
df_knn = pd.concat([X_imputed_knn, y], axis=1)

print("Sprawdzenie braków po imputacji KNN:")
print(df_knn.isnull().sum().head(5))

Sprawdzenie braków po imputacji KNN:
having_IP_Address           0
URL_Length                  0
Shortining_Service          0
having_At_Symbol            0
double_slash_redirecting    0
dtype: int64


In [6]:
outliers_mask = ~df_knn.isin([-1, 0, 1, -1.0, 0.0, 1.0])

outliers_count = outliers_mask.sum().sum()
print(f"Liczba wykrytych klasycznych wartości odstających (poza zbiorem -1, 0, 1): {outliers_count}")

for col in df_knn.columns[:5]:
    print(f"Unikalne wartości w kolumnie {col}: {df_knn[col].unique()}")

Liczba wykrytych klasycznych wartości odstających (poza zbiorem -1, 0, 1): 0
Unikalne wartości w kolumnie having_IP_Address: [-1.  1.  0.]
Unikalne wartości w kolumnie URL_Length: [ 1.  0. -1.]
Unikalne wartości w kolumnie Shortining_Service: [ 1. -1.  0.]
Unikalne wartości w kolumnie having_At_Symbol: [ 1. -1.  0.]
Unikalne wartości w kolumnie double_slash_redirecting: [-1.  1.  0.]


In [7]:
import h2o

print("Uruchamianie Auto ML (H2O) do imputacji braków...")

h2o.init(verbose=False)

hf_missing = h2o.H2OFrame(df_missing)

for col in hf_missing.columns:
    if col != 'Result':
        hf_missing.impute(col, method='mode')

df_automl = hf_missing.as_data_frame()
df_automl = df_automl[df_missing.columns]

h2o.cluster().shutdown()


base_datasets = {
    'mode': df_mode,        # Prosty 1
    'const': df_const,      # Prosty 2
    'knn': df_knn,          # Model
    'automl': df_automl     # Auto ML (H2O)
}

def min_max_scaler_manual(df, target_col='Result'):
    df_scaled = df.copy()
    features = [c for c in df.columns if c != target_col]

    for col in features:
        min_val = df_scaled[col].min()
        max_val = df_scaled[col].max()
        if max_val != min_val:
            df_scaled[col] = (df_scaled[col] - min_val) / (max_val - min_val)

    return df_scaled

def standardization_manual(df, target_col='Result'):
    df_scaled = df.copy()
    features = [c for c in df.columns if c != target_col]

    for col in features:
        mean_val = df_scaled[col].mean()
        std_val = df_scaled[col].std()
        if std_val != 0:
            df_scaled[col] = (df_scaled[col] - mean_val) / std_val

    return df_scaled

final_datasets = {}

for name, dataset in base_datasets.items():
    final_datasets[f"{name}_minmax"] = min_max_scaler_manual(dataset)
    final_datasets[f"{name}_std"] = standardization_manual(dataset)

print(f"\nPomyślnie wygenerowano {len(final_datasets)} przeskalowanych zbiorów danych!")
print("Lista utworzonych zbiorów:")
for name in final_datasets.keys():
    print(f"- {name}")

print("\nPrzykład: 5 pierwszych wierszy dla 'automl_std' (Standaryzacja z H2O):")
display(final_datasets['automl_std'].head())

Uruchamianie Auto ML (H2O) do imputacji braków...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


C:\Users\User\PycharmProjects\ML_Phishing_Project\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



Pomyślnie wygenerowano 8 przeskalowanych zbiorów danych!
Lista utworzonych zbiorów:
- mode_minmax
- mode_std
- const_minmax
- const_std
- knn_minmax
- knn_std
- automl_minmax
- automl_std

Przykład: 5 pierwszych wierszy dla 'automl_std' (Standaryzacja z H2O):


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1.496924,2.274479,0.370427,0.396197,-2.756036,-0.369795,-1.402951,-1.480113,-0.650045,0.450757,...,0.460167,0.303057,-1.166423,-1.598259,-1.671273,-0.544745,0.377498,1.255635,-2.632449,-1
1,0.667976,2.274479,0.370427,0.396197,0.362807,-0.369795,-0.190599,0.755680,-0.650045,0.450757,...,0.460167,0.303057,-1.166423,-1.598259,0.788084,-0.544745,0.377498,1.255635,0.379840,-1
2,0.667976,-0.448393,0.370427,0.396197,0.362807,-0.369795,-1.402951,-1.480113,-0.650045,0.450757,...,0.460167,0.303057,0.857244,-1.598259,0.788084,-0.544745,0.377498,-0.561731,-2.632449,-1
3,0.667976,0.913043,0.370427,0.396197,0.362807,-0.369795,-1.402951,-1.480113,1.538216,0.450757,...,0.460167,0.303057,-1.166423,-1.598259,0.788084,-0.544745,0.377498,-2.379098,0.379840,-1
4,0.667976,0.913043,-2.699342,0.396197,0.362807,-0.369795,1.021754,0.755680,-0.650045,0.450757,...,-2.172929,0.303057,-1.166423,-1.598259,-0.441594,-0.544745,0.377498,1.255635,0.379840,1


In [8]:
url_features = ['URL_Length', 'having_At_Symbol', 'Prefix_Suffix', 'Abnormal_URL']

df_automl['URL_Risk_Score'] = df_automl[url_features].sum(axis=1)

print("Wygląd nowej cechy 'URL_Risk_Score' obok cech składowych (5 pierwszych wierszy):")
display(df_automl[url_features + ['URL_Risk_Score', 'Result']].head())

korelacja = df_automl['URL_Risk_Score'].corr(df_automl['Result'])
print(f"\nKorelacja nowej cechy (URL_Risk_Score) z wynikiem (Result) wynosi: {korelacja:.2f}")

Wygląd nowej cechy 'URL_Risk_Score' obok cech składowych (5 pierwszych wierszy):


,URL_Length,having_At_Symbol,Prefix_Suffix,Abnormal_URL,URL_Risk_Score,Result
0,1,1,-1,-1,0,-1
1,1,1,-1,1,2,-1
2,-1,1,-1,-1,-2,-1
3,0,1,-1,1,1,-1
4,0,1,-1,1,1,1



Korelacja nowej cechy (URL_Risk_Score) z wynikiem (Result) wynosi: 0.19
